# Session 14: SQL Optimization & AI Performance Tuning

> Eliminate N+1 queries, design optimal indexes, and use AI to interpret execution plans.

## 1. The N+1 Problem

**The N+1 problem is one of the most common performance issues in database-backed applications.** It happens when code fetches a list of N records and then executes one additional query *per record* to load related data — resulting in N+1 total queries instead of 1 or 2.

**How it happens:**
```python
# Looks innocent in Python, but causes N+1 queries
orders = session.query(Order).all()   # Query 1: SELECT * FROM orders
for order in orders:
    print(order.customer.name)         # Query 2…N+1: SELECT * FROM customers WHERE id = ?
```
If there are 1,000 orders, this executes 1,001 queries. Each query has network round-trip overhead, connection pool overhead, and parser overhead. A page that should load in 50ms takes 5 seconds.

**Why it's invisible locally:** Your local database is on `localhost` — each query takes <1ms. In production, the database is a separate server — each query takes 2–5ms. 1,000 queries × 3ms = 3 seconds.

**The fix — eager loading:**
```python
# SQLAlchemy: load customers in a single JOIN
orders = session.query(Order).options(joinedload(Order.customer)).all()
# Now: SELECT orders.*, customers.* FROM orders JOIN customers ON ...
```
ORMs provide `joinedload` (JOIN), `subqueryload` (separate query with IN clause), and `selectinload` (separate query per batch). Choose based on the cardinality of the relationship.

**Detecting N+1 in production:** Use query count logging in development. Django Debug Toolbar, SQLAlchemy event listeners, and the `nplusone` library can automatically detect and warn about N+1 patterns during tests.

In [ ]:
# ❌ N+1: 1 query for orders + N queries for users
orders = [{'id':1,'user_id':10},{'id':2,'user_id':11},{'id':3,'user_id':10}]
for order in orders:
    # This fires a query for EVERY order
    user = f'SELECT * FROM users WHERE id={order["user_id"]}'
    print(f'Order {order["id"]} by user {order["user_id"]}')
print(f'Total queries: {1 + len(orders)} for {len(orders)} orders ← N+1!')

print('\n✅ Fix 1: JOIN')
fix_join = '''
SELECT o.id, o.status, u.name, u.email
FROM orders o
INNER JOIN users u ON u.id = o.user_id
WHERE o.status = 'pending'
'''
print(fix_join)

print('✅ Fix 2: WHERE IN (batch fetch)')
user_ids = list(set(o['user_id'] for o in orders))
batch_query = f'SELECT * FROM users WHERE id IN ({','.join(map(str,user_ids))})'
print(batch_query, '← 2 queries total, regardless of N')

## 2. Index Strategy & EXPLAIN ANALYZE

**An index is a data structure (typically a B-tree) that allows the database to find rows matching a condition without scanning every row in the table.** Without an index, the database performs a *sequential scan* — reading every row to find matches. On a table with 10 million rows, a sequential scan takes seconds. With an index, the same lookup takes milliseconds.

**When to add an index:**
1. **Foreign key columns:** Always index FKs. JOINs without indexes on FK columns cause full table scans.
2. **Columns in WHERE clauses of frequent queries:** If you always filter by `status`, index `status`.
3. **Columns in ORDER BY:** The database can use an index to return pre-sorted results without a sort step.
4. **Unique constraints:** Enforced via an index automatically.

**When indexes hurt:**
- **Write performance:** Every INSERT, UPDATE, DELETE must update all indexes on the table. A table with 10 indexes is 10× slower to write to.
- **Storage:** Indexes consume disk space — sometimes more than the table itself.
- **Over-indexing:** Indexes on low-cardinality columns (e.g., `is_active BOOLEAN` with 95% `true`) may not be used at all — the optimizer may decide a sequential scan is faster.

---

**EXPLAIN ANALYZE — reading query plans:**

`EXPLAIN ANALYZE` shows you *what the database actually did* to execute a query. Key terms to know:
- **Seq Scan:** Sequential scan — no index used. Acceptable on small tables, a red flag on large ones.
- **Index Scan:** Used the index to find rows, then fetched the actual rows from the table.
- **Index Only Scan:** All needed columns were in the index itself — fastest, no table access needed.
- **Nested Loop / Hash Join / Merge Join:** Different strategies for combining two tables. Hash Join is generally fast for large tables.
- **cost=X..Y:** Estimated startup cost and total cost. Lower is better.
- **actual time=X..Y rows=Z:** The real measured time and row count — compare to the estimate to spot miscalibrated statistics.

**Practical workflow:** Run `EXPLAIN ANALYZE` on any query that takes more than ~50ms. Find the most expensive node in the plan. Add an index for the column being scanned, then run again to verify the plan changed.

In [ ]:
index_guide = {
    'Always index':    ['Primary keys (auto)','Foreign keys','WHERE clause columns (high cardinality)'],
    'Consider':        ['ORDER BY columns','JOIN ON columns','Partial indexes for filtered queries'],
    'Avoid':           ['Low cardinality cols (status, boolean)','Tables < 1000 rows','Columns updated frequently'],
}
for category, items in index_guide.items():
    print(f'\n{category}:')
    for i in items: print(f'  • {i}')

explain_example = '''
-- BEFORE index
Seq Scan on orders (rows=50000, cost=0..1240, time=287ms) ← SLOW
  Filter: status = 'processing'

-- AFTER: CREATE INDEX CONCURRENTLY idx_orders_status ON orders(status)
Index Scan using idx_orders_status (rows=23, cost=0..8, time=0.4ms) ← 700x faster
'''
print(explain_example)

## 💡 Additional Examples: SQL Optimization

**Example 1 — Pagination: OFFSET vs keyset (cursor-based)**
`LIMIT 20 OFFSET 10000` is a performance trap. The database must read and discard 10,000 rows before returning the 20 you want. On page 500 of a result set, this means discarding 10,000 rows on every request. Keyset pagination (`WHERE id > last_seen_id ORDER BY id LIMIT 20`) jumps directly to the right position using the index. It's O(log n) instead of O(n). The trade-off: you can't jump to an arbitrary page number — keyset pagination only supports "next page". For most real-world UIs (infinite scroll, "load more"), this is perfectly acceptable.

**Example 2 — Covering indexes for read-heavy queries**
If a query always selects the same small set of columns (e.g., `SELECT id, email FROM users WHERE status = 'active'`), you can create a **covering index** that includes all needed columns: `CREATE INDEX idx_users_status_email ON users(status, email)`. The database can satisfy the entire query from the index without ever touching the main table (an Index Only Scan). This is one of the most impactful optimizations for read-heavy queries.

**Example 3 — Query result caching**
Not every slow query can be optimized further. If a query is inherently expensive (e.g., an aggregate over millions of rows for a dashboard) and the data doesn't change every second, caching the result is the right answer. Store the result in Redis with a TTL (e.g., 60 seconds). The first request in each window pays the full cost; all subsequent requests are served from cache in <1ms. The key design decision is the cache invalidation strategy: time-based TTL (simple, may serve stale data), or event-based invalidation (more complex, always fresh).

In [ ]:
# ─── Example 1: Query Rewriting — Subquery vs CTE vs Window Functions ─────────
query_comparisons = [
    {
        'scenario': 'Find top 3 products by revenue per category',
        'bad': '''
-- ❌ Correlated subquery — O(n²) complexity, runs once per row
SELECT p.name, p.category, p.revenue
FROM products p
WHERE (
    SELECT COUNT(*) FROM products p2
    WHERE p2.category = p.category
    AND p2.revenue > p.revenue
) < 3
ORDER BY p.category, p.revenue DESC;
        ''',
        'good': '''
-- ✅ Window function — single pass O(n log n)
WITH ranked AS (
    SELECT
        name,
        category,
        revenue,
        ROW_NUMBER() OVER (
            PARTITION BY category
            ORDER BY revenue DESC
        ) AS rank
    FROM products
)
SELECT name, category, revenue
FROM ranked
WHERE rank <= 3
ORDER BY category, rank;
        ''',
        'improvement': '~80% faster on 1M+ rows — avoids correlated subquery re-scan',
    },
    {
        'scenario': 'Monthly revenue with running total',
        'bad': '''
-- ❌ Self JOIN — creates n² combinations then aggregates
SELECT m.month, m.revenue,
    (SELECT SUM(m2.revenue) FROM monthly_revenue m2
     WHERE m2.month <= m.month) AS running_total
FROM monthly_revenue m
ORDER BY m.month;
        ''',
        'good': '''
-- ✅ Window SUM — single scan with frame
SELECT
    month,
    revenue,
    SUM(revenue) OVER (ORDER BY month
                       ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS running_total,
    AVG(revenue) OVER (ORDER BY month
                       ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
    ) AS moving_avg_3m
FROM monthly_revenue
ORDER BY month;
        ''',
        'improvement': 'Single pass vs N separate subqueries',
    },
]

for i, qc in enumerate(query_comparisons, 1):
    print(f'=== Scenario {i}: {qc["scenario"]} ===')
    print(qc['bad'])
    print(qc['good'])
    print(f'  💡 Improvement: {qc["improvement"]}\n')

# ─── Example 2: Composite Index Strategy ──────────────────────────────────────
index_scenarios = [
    {
        'table': 'orders',
        'query': "WHERE user_id = $1 AND status = 'processing' ORDER BY created_at DESC",
        'bad_index': 'CREATE INDEX idx_orders_status ON orders(status);',
        'bad_issue': 'Low cardinality index on status — still scans thousands of rows per user',
        'good_index': 'CREATE INDEX idx_orders_user_status_created ON orders(user_id, status, created_at DESC);',
        'good_reason': 'Composite index: filters user_id first (high cardinality), then status, covers ORDER BY',
    },
    {
        'table': 'audit_logs',
        'query': "WHERE table_name = 'orders' AND changed_at >= NOW() - INTERVAL '7 days'",
        'bad_index': 'CREATE INDEX idx_audit_table ON audit_logs(table_name);',
        'bad_issue': 'Index only on table_name — still scans all historical rows for that table',
        'good_index': "CREATE INDEX idx_audit_table_date ON audit_logs(table_name, changed_at DESC) WHERE changed_at > '2026-01-01';",
        'good_reason': 'Partial index: excludes old data, keeps the index small; changed_at filters efficiently',
    },
    {
        'table': 'products',
        'query': "WHERE category_id = $1 AND is_active = TRUE AND price BETWEEN $2 AND $3",
        'bad_index': 'CREATE INDEX idx_products_price ON products(price);',
        'bad_issue': 'Wrong leading column — price scan returns millions of rows, then filters category',
        'good_index': 'CREATE INDEX idx_products_cat_active_price ON products(category_id, is_active, price);',
        'good_reason': 'category_id leads (most selective), is_active narrows further, price enables range scan',
    },
]

print('=== Composite Index Design Guide ===\n')
for s in index_scenarios:
    print(f'Table: {s["table"]}')
    print(f'Query:  {s["query"]}')
    print(f'  ❌ Bad:  {s["bad_index"]}')
    print(f'          Issue: {s["bad_issue"]}')
    print(f'  ✅ Good: {s["good_index"]}')
    print(f'          Why:   {s["good_reason"]}')
    print()

# ─── Example 3: N+1 Query Detection and Fix ───────────────────────────────────
# Simulates Django ORM-style query tracking to illustrate the N+1 problem

class QueryTracker:
    def __init__(self): self.queries = []
    def record(self, sql: str, rows: int = 1):
        self.queries.append({'sql': sql, 'rows': rows})
        return [{'id': i} for i in range(rows)]

tracker = QueryTracker()

# ❌ N+1 Pattern: one query to fetch orders, then one query PER order to fetch its user
print('=== N+1 Anti-pattern ===')
orders = tracker.record('SELECT * FROM orders WHERE status=pending', rows=50)  # 1 query
for order in orders:
    user = tracker.record(f'SELECT * FROM users WHERE id={order["id"]}')       # 50 queries!

n_plus_1_total = len(tracker.queries)
print(f'N+1 total queries: {n_plus_1_total} (1 + {len(orders)} orders)')

# Reset
tracker.queries.clear()

# ✅ Fix: use JOIN / select_related / eager loading
print('\n=== Optimized with JOIN ===')
orders_with_users = tracker.record(
    'SELECT o.*, u.* FROM orders o INNER JOIN users u ON u.id=o.user_id WHERE o.status=pending',
    rows=50
)
optimized_total = len(tracker.queries)
print(f'Optimized total queries: {optimized_total} (single JOIN)')
print(f'Query reduction: {n_plus_1_total} → {optimized_total} ({(1 - optimized_total/n_plus_1_total)*100:.0f}% fewer queries)')

# ORM equivalents across popular frameworks
orm_fixes = {
    'Django (FK)':         "Order.objects.select_related('user').filter(status='pending')",
    'Django (M2M)':        "Order.objects.prefetch_related('tags').filter(status='pending')",
    'SQLAlchemy':          "session.query(Order).options(joinedload(Order.user)).filter_by(status='pending')",
    'Eloquent (Laravel)':  "Order::with('user')->where('status', 'pending')->get()",
    'Prisma (Node.js)':    "prisma.order.findMany({ where: { status: 'pending' }, include: { user: true } })",
}
print('\nORM equivalents to fix N+1:')
for orm, code in orm_fixes.items():
    print(f'  {orm:<25} {code}')


## 3. AI Lab: Query Optimization

### 🧪 AI Lab / Practice

> **TODO:** Find your 3 slowest queries using pg_stat_statements or slow query log → For each, paste query + EXPLAIN ANALYZE output to Claude: 'Analyze this query plan. Identify the bottleneck, suggest specific index creation, and rewrite the query if needed. Show before/after execution time.' → Implement top suggestions and re-measure.

In [ ]:
# TODO: Implement your solution here
raise NotImplementedError('Not implemented yet — complete this exercise')